# Coordinate check: original vs. parameter grid
- Loads `coordinatesAll_original.csv` and `coordinatesAll_param.csv` as separate tables.
- Narrows the param table to the settings the original run used, then checks every overlapping row has the same peak coordinates.

In [1]:
import pandas as pd

origFile = 'coordinatesAll_original.csv'
paramFile = 'coordinatesAll_param.csv'

# Settings of the original run (serverFunctions/analysisConfig.py): the base run tag 'param' is
# blur 4 / motion 0.3 / outliers 0.15, and 'abs' is peakScript.py's original |z| peak search.
origRunTag = 'param'        # 'param-b4m0p3' rebuilds the same settings through x07; swap in to check it too
origSign = 'abs'

keyCols = ['subject', 'session', 'task', 'sequence_type', 'target', 'clustRegion', 'clustMask']
coordCols = ['X', 'Y', 'Z']
coordTol = 0                # mm; coordinates are voxel centers, so expect an exact match

## Load

In [2]:
# subject/session as strings so leading zeros ('001', '02') survive
readOpts = dict(dtype={'subject': str, 'session': str})
origDf = pd.read_csv(origFile, **readOpts)
paramDf = pd.read_csv(paramFile, **readOpts)

print(f"original: {origDf.shape}, columns {list(origDf.columns)}")
print(f"param:    {paramDf.shape}, columns {list(paramDf.columns)}")
for col in ['sequence_type', 'runTag', 'sign', 'clustMask', 'target']:
    if col in paramDf:
        print(f"\nparam {col}:\n{paramDf[col].value_counts().to_string()}")

original: (6264, 12), columns ['filename', 'filepath', 'subject', 'session', 'task', 'sequence_type', 'target', 'clustRegion', 'clustMask', 'X', 'Y', 'Z']
param:    (81366, 15), columns ['filename', 'filepath', 'subject', 'session', 'task', 'sequence_type', 'runTag', 'target', 'sign', 'clustRegion', 'clustMask', 'X', 'Y', 'Z', 'peakZ']

param sequence_type:
sequence_type
me    81366

param runTag:
runTag
param-b4m0p3     6246
param-b0m0p3     6246
param-b0m0p45    6246
param-b4m0p45    6246
param-b2m0p3     6246
param-b2m0p45    6246
param            6246
param-b6m0p3     6246
param-b6m0p45    6246
param-b4m0p15    6228
param-b0m0p15    6228
param-b2m0p15    6228
param-b6m0p15    6228

param sign:
sign
abs    27282
pos    27042
neg    27042

param clustMask:
clustMask
orig        27122
dilate_5    27122
erode_1     27122

param target:
target
seed       40683
network    40683


## Narrow the param table to the original settings

In [3]:
paramOrigDf = paramDf[(paramDf['runTag'] == origRunTag) & (paramDf['sign'] == origSign)].copy()
print(f"param rows at runTag='{origRunTag}', sign='{origSign}': {len(paramOrigDf)} of {len(paramDf)}")

# Each key should name one row in each table, or the merge below would pair rows arbitrarily
for name, df in [('original', origDf), ('param (narrowed)', paramOrigDf)]:
    nDup = df.duplicated(keyCols).sum()
    print(f"{name}: {nDup} duplicated keys")
    assert nDup == 0, f"{name} has duplicated keys"


param rows at runTag='param', sign='abs': 2082 of 81366
original: 0 duplicated keys
param (narrowed): 0 duplicated keys


## Compare overlapping rows

In [4]:
merged = origDf.merge(paramOrigDf, on=keyCols, how='outer', suffixes=('_orig', '_param'), indicator=True)
print(merged['_merge'].value_counts().rename({'left_only': 'original only', 'right_only': 'param only'}).to_string())

overlap = merged[merged['_merge'] == 'both'].copy()
diffs = pd.DataFrame({c: (overlap[f'{c}_orig'] - overlap[f'{c}_param']).abs() for c in coordCols})
overlap['maxDiff'] = diffs.max(axis=1)
mismatch = overlap[overlap['maxDiff'] > coordTol]

print(f"\noverlapping rows: {len(overlap)}")
print(f"matching:         {len(overlap) - len(mismatch)}")
print(f"mismatched:       {len(mismatch)}")

_merge
original only    4182
both             2082
param only          0

overlapping rows: 2082
matching:         2076
mismatched:       6


In [5]:
# Mismatched rows, side by side
showCols = keyCols + [f'{c}_{s}' for c in coordCols for s in ('orig', 'param')] + ['peakZ', 'maxDiff']
mismatch[showCols].sort_values('maxDiff', ascending=False)

,subject,session,task,sequence_type,target,clustRegion,clustMask,X_orig,X_param,Y_orig,Y_param,Z_orig,Z_param,peakZ,maxDiff
93,003,02,rest,me,seed,ldlpfc,dilate_5,19.0,61.0,-19.0,-17.0,41.0,21.0,0.1283,42.0
92,003,02,rest,me,network,ldlpfc,orig,47.0,43.0,-39.0,-21.0,25.0,25.0,0.1639,18.0
95,003,02,rest,me,seed,ldlpfc,orig,45.0,41.0,-25.0,-29.0,31.0,39.0,0.0921,8.0
90,003,02,rest,me,network,ldlpfc,dilate_5,27.0,21.0,-23.0,-31.0,49.0,47.0,0.4718,8.0
91,003,02,rest,me,network,ldlpfc,erode_1,41.0,35.0,-23.0,-29.0,37.0,31.0,0.1300,6.0
94,003,02,rest,me,seed,ldlpfc,erode_1,45.0,41.0,-25.0,-25.0,31.0,29.0,0.0742,4.0


In [6]:
# Where mismatches fall, as a fraction of overlapping rows per group
if len(overlap):
    overlap['mismatch'] = overlap['maxDiff'] > coordTol
    display(overlap.groupby(['target', 'clustMask'])['mismatch'].agg(['sum', 'count', 'mean']))
    display(overlap.groupby('subject')['mismatch'].sum().loc[lambda s: s > 0].sort_values(ascending=False))

sum  count      mean
target  clustMask                      
network dilate_5     1    347  0.002882
        erode_1      1    347  0.002882
        orig         1    347  0.002882
seed    dilate_5     1    347  0.002882
        erode_1      1    347  0.002882
        orig         1    347  0.002882

subject
003    6
Name: mismatch, dtype: int64

## Rows without a partner

In [7]:
# Original-only rows: expected for sequence types the param grid didn't run (se, se_e2)
origOnly = merged[merged['_merge'] == 'left_only']
print("original only, by sequence_type:")
print(origOnly['sequence_type'].value_counts().to_string())

# Among sequence types both tables have, any unpaired row is a gap in one run or the other
sharedSeq = set(origDf['sequence_type']) & set(paramOrigDf['sequence_type'])
unpaired = merged[(merged['_merge'] != 'both') & merged['sequence_type'].isin(sharedSeq)]
print(f"\nunpaired rows within shared sequence types {sorted(sharedSeq)}: {len(unpaired)}")
unpaired[keyCols + ['_merge']].sort_values(keyCols)

original only, by sequence_type:
sequence_type
se       2088
se_e2    2088
me          6

unpaired rows within shared sequence types ['me']: 6


,subject,session,task,sequence_type,target,clustRegion,clustMask,_merge
5148,083,03,rest,me,network,ldlpfc,dilate_5,left_only
5149,083,03,rest,me,network,ldlpfc,erode_1,left_only
5150,083,03,rest,me,network,ldlpfc,orig,left_only
5151,083,03,rest,me,seed,ldlpfc,dilate_5,left_only
5152,083,03,rest,me,seed,ldlpfc,erode_1,left_only
5153,083,03,rest,me,seed,ldlpfc,orig,left_only


In [8]:
assert len(mismatch) == 0, f"{len(mismatch)} overlapping rows differ"
print(f"All {len(overlap)} overlapping rows match (tolerance {coordTol} mm).")

AssertionError: 6 overlapping rows differ